In [15]:
# -*- coding: utf-8 -*-
"""Bài tập 1: Tự viết KNN không dùng thư viện."""

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


def knn_du_doan(X_hoc, y_hoc, x_moi, k):
  """Dự đoán nhãn cho một mẫu mới x_moi dựa trên k láng giềng gần nhất."""
  # Tính khoảng cách Euclid từ x_moi đến mọi mẫu trong X_hoc
  khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))
  # Lấy chỉ số của k khoảng cách nhỏ nhất
  idx = np.argsort(khoang_cach)[:k]
  phieu = y_hoc[idx]
  # Bỏ phiếu đa số (với bài toán 2 lớp)
  so_1 = (phieu == 1).sum()
  return 1 if so_1 >= (k / 2) else 0


if __name__ == "__main__":
  df = pd.read_csv("hai_vang_trang.csv")
  X = df[["x1", "x2"]].to_numpy()
  y = df["nhan"].to_numpy()

  X_tr, X_te, y_tr, y_te = train_test_split(
      X, y, test_size=0.25, random_state=17, stratify=y
  )

  # Dự đoán cho toàn bộ tập kiểm tra với k = 13
  k = 13
  y_pred = [knn_du_doan(X_tr, y_tr, x, k) for x in X_te]
  acc = accuracy_score(y_te, y_pred)

  print(f"Accuracy tu viet (k=13): {acc:.4f}")

Accuracy tu viet (k=13): 0.9200


In [16]:
# -*- coding: utf-8 +*-
"""Bài tập 2: Sử dụng khoảng cách Manhattan cho KNN."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

if __name__ == "__main__":
  df = pd.read_csv("hai_vang_trang.csv")
  X = df[["x1", "x2"]]
  y = df["nhan"]

  X_tr, X_te, y_tr, y_te = train_test_split(
      X, y, test_size=0.25, random_state=17, stratify=y
  )

  diem = {}
  for k in range(1, 52, 2):
    scores = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, metric="manhattan"),
        X_tr,
        y_tr,
        cv=5,
    )
    diem[k] = scores.mean()

  k_tot = max(diem, key=diem.get)
  mh = KNeighborsClassifier(n_neighbors=k_tot, metric="manhattan").fit(
      X_tr, y_tr
  )
  acc = accuracy_score(y_te, mh.predict(X_te))

  print(f"K tot nhat (Manhattan) = {k_tot}")
  print(f"Accuracy tren tap kiem tra = {acc:.4f}")

K tot nhat (Manhattan) = 27
Accuracy tren tap kiem tra = 0.9067


In [17]:
# -*- coding: utf-8 -*-
"""Bài tập 3: Đổi đơn vị đo cột chi tiêu."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

if __name__ == "__main__":
  df = pd.read_csv("sinh_vien_mo_rong.csv")
  # Đổi đơn vị chi tiêu từ nghìn sang triệu đồng
  df["chi_tieu_trieu"] = df["chi_tieu"] / 1000.0

  X = df[["gio_on", "diem_giua_ky", "chi_tieu_trieu"]]
  y = df["qua_mon"]

  X_tr, X_te, y_tr, y_te = train_test_split(
      X, y, test_size=0.25, random_state=17, stratify=y
  )

  mh = KNeighborsClassifier(n_neighbors=7)
  mh.fit(X_tr, y_tr)
  acc = accuracy_score(y_te, mh.predict(X_te))

  print(f"Accuracy khi doi don vi (khong chuan hoa): {acc:.4f}")

Accuracy khi doi don vi (khong chuan hoa): 0.8667


In [18]:
# -*- coding: utf-8 -*-
"""Bài tập 4: KNN với trọng số theo khoảng cách (weights='distance')."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

if __name__ == "__main__":
  df = pd.read_csv("hai_vang_trang.csv")
  X = df[["x1", "x2"]]
  y = df["nhan"]

  X_tr, X_te, y_tr, y_te = train_test_split(
      X, y, test_size=0.25, random_state=17, stratify=y
  )

  for k in [1, 13, 101]:
    mh = KNeighborsClassifier(n_neighbors=k, weights="distance")
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f"k = {3:3d} (weights='distance') -> Accuracy: {acc:.4f}" if k==1 else f"k = {k:3d} (weights='distance') -> Accuracy: {acc:.4f}")

k =   3 (weights='distance') -> Accuracy: 0.8000
k =  13 (weights='distance') -> Accuracy: 0.9333
k = 101 (weights='distance') -> Accuracy: 0.9067


In [19]:
# -*- coding: utf-8 -*-
"""Bài tập 5: Chọn k tốt nhất cho KNN Hồi quy trên dữ liệu giá nhà."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

if __name__ == "__main__":
  df = pd.read_csv("gia_nha.csv")
  X = df[["dien_tich"]]
  y = df["gia"]

  X_tr, X_te, y_tr, y_te = train_test_split(
      X, y, test_size=0.2, random_state=42
  )

  mae_dict = {}
  for k in range(1, 21):
    scores = cross_val_score(
        KNeighborsRegressor(n_neighbors=k),
        X_tr,
        y_tr,
        cv=5,
        scoring="neg_mean_absolute_error",
    )
    mae_dict[k] = -scores.mean()   # đổi dấu âm thành dương

  k_tot = min(mae_dict, key=mae_dict.get)
  print(f"K tot nhat cho hoi quy = {k_tot}")

  # Đánh giá trên tập kiểm tra
  knn_reg = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
  lin_reg = LinearRegression().fit(X_tr, y_tr)

  mae_knn = mean_absolute_error(y_te, knn_reg.predict(X_te))
  mae_lin = mean_absolute_error(y_te, lin_reg.predict(X_te))

  print(f"MAE KNN (k={k_tot}) : {mae_knn:.4f} tỷ")
  print(f"MAE Duong thang  : {mae_lin:.4f} tỷ")

K tot nhat cho hoi quy = 2
MAE KNN (k=2) : 0.4867 tỷ
MAE Duong thang  : 0.2578 tỷ


In [20]:
# -*- coding: utf-8 -*-
"""Bài tập 6: Đo thời gian huấn luyện và dự đoán của KNN theo quy mô dữ liệu."""

import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

if __name__ == "__main__":
  rng = np.random.default_rng(0)
  sizes = [1000, 10000, 100000]

  print(f"{'Cỡ dữ liệu':<12} | {'Thời gian fit (s)':<18} | {'Thời gian predict (s)'}")
  print("-" * 55)

  for sz in sizes:
    X_hoc = rng.random((sz, 2))
    y_hoc = rng.integers(0, 2, size=sz)
    X_moi = rng.random((1000, 2))

    # Đo thời gian fit
    t0 = time.time()
    mh = KNeighborsClassifier(n_neighbors=5).fit(X_hoc, y_hoc)
    t_fit = time.time() - t0

    # Đo thời gian predict
    t0 = time.time()
    _ = mh.predict(X_moi)
    t_pred = time.time() - t0

    print(f"{sz:<12} | {t_fit:<18.6f} | {t_pred:.6f}")

Cỡ dữ liệu   | Thời gian fit (s)  | Thời gian predict (s)
-------------------------------------------------------
1000         | 0.002671           | 0.008976
10000        | 0.005656           | 0.005166
100000       | 0.069695           | 0.008014
